In [0]:
# ============================================================
# GOLD LAYER - FACT_DELINQUENCY - UPDATED FOR CI/CD
# ============================================================

import os
from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# LOAD CONFIGURATION FROM ENVIRONMENT VARIABLES
# ============================================================

# Get environment (DEV, TEST, PROD) - Default to DEV
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from widget: {env}")
except:
    env = os.getenv('ENV', 'DEV')
    print(f"📌 Environment from env var: {env}")

# Get storage account and access key from environment (GitHub Secrets)
storage_account_name = os.getenv('STORAGE_ACCOUNT', 'capstonestorage01')
access_key = os.getenv('STORAGE_ACCESS_KEY')

# Set container names based on environment
if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# Configure Spark with access key (from GitHub Secrets)
if access_key:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        access_key
    )
    print("✅ Storage access key configured from GitHub Secrets")
else:
    print("⚠️ WARNING: No access key found! Using Azure AD authentication")

# Dynamic paths based on environment
SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD   = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

# ============================================================
# YOUR EXISTING CODE (NO CHANGES NEEDED BELOW!)
# ============================================================

# ── Load ──
df_deliq     = spark.read.format("delta").load(f"{SILVER}/repayment_delinquency")
dim_account  = spark.read.format("delta").load(f"{GOLD}/dim_account")
dim_customer = spark.read.format("delta").load(f"{GOLD}/dim_customer")

# ── Join & Transform ──
fact_delinquency = df_deliq\
    .join(dim_account.select(
            "account_id","customer_id","product_type",
            "credit_limit","credit_tier"),
          "account_id", "left")\
    .join(dim_customer.select(
            "customer_id","risk_segment","city","income_band"),
          "customer_id", "left")\
    .select(
        F.col("account_id"),
        F.col("customer_id"),
        F.col("due_date"),
        F.col("payment_date"),
        F.col("days_past_due"),
        F.col("overdue_amount"),
        F.col("delinquency_flag"),
        F.col("product_type"),
        F.col("credit_limit"),
        F.col("credit_tier"),
        F.col("risk_segment"),
        F.col("city"),
        F.col("income_band"),
        F.when(F.col("days_past_due") == 0,  F.lit("Current"))
         .when(F.col("days_past_due") <= 30, F.lit("DPD 1-30"))
         .when(F.col("days_past_due") <= 60, F.lit("DPD 31-60"))
         .when(F.col("days_past_due") <= 90, F.lit("DPD 61-90"))
         .otherwise(F.lit("DPD 90+"))
         .alias("dpd_bucket"),
        F.round(F.col("overdue_amount") / F.col("credit_limit") * 100, 2)
         .alias("overdue_pct_of_limit"),
        F.datediff(F.col("payment_date"), F.col("due_date"))
         .alias("actual_delay_days"),
        F.current_timestamp().alias("gold_created_at")
    )

# ── Write ──
fact_delinquency.write\
    .format("delta")\
    .mode("overwrite")\
    .partitionBy("product_type")\
    .save(f"{GOLD}/fact_delinquency")

print(f"\n{'='*55}")
print(f"✅ fact_delinquency : {fact_delinquency.count():,} rows")
print(f"📁 Written to: {GOLD}/fact_delinquency")
print(f"🏭 Environment: {env}")
print(f"{'='*55}")

✅ fact_delinquency : 200,000 rows
